# Stock picker: structured research and a manager: practice

StockPicker uses a hierarchical manager, typed research lists, search tools, notification tool, and persisted memory. Practice makes no investment recommendation or notification.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [3_crew/stock_picker/pyproject.toml](../../../../../3_crew/stock_picker/pyproject.toml)
- [3_crew/stock_picker/README.md](../../../../../3_crew/stock_picker/README.md)
- [3_crew/stock_picker/report.md](../../../../../3_crew/stock_picker/report.md)
- [3_crew/stock_picker/src/stock_picker/__init__.py](../../../../../3_crew/stock_picker/src/stock_picker/__init__.py)
- [3_crew/stock_picker/src/stock_picker/config/agents.yaml](../../../../../3_crew/stock_picker/src/stock_picker/config/agents.yaml)
- [3_crew/stock_picker/src/stock_picker/config/tasks.yaml](../../../../../3_crew/stock_picker/src/stock_picker/config/tasks.yaml)
- [3_crew/stock_picker/src/stock_picker/crew.py](../../../../../3_crew/stock_picker/src/stock_picker/crew.py)
- [3_crew/stock_picker/src/stock_picker/main.py](../../../../../3_crew/stock_picker/src/stock_picker/main.py)
- [3_crew/stock_picker/src/stock_picker/tools/__init__.py](../../../../../3_crew/stock_picker/src/stock_picker/tools/__init__.py)
- [3_crew/stock_picker/src/stock_picker/tools/push_tool.py](../../../../../3_crew/stock_picker/src/stock_picker/tools/push_tool.py)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Roles, tasks, and process

CrewAI separates who does work from what work is required. A sequential process has a fixed task order; a hierarchical process gives a manager coordination responsibility.

**Predict:** Is a role description itself an executable task?

In [ ]:
agents = {"researcher": "Collect fictional evidence", "analyst": "Interpret supplied evidence"}
tasks = [{"name": "research_task", "agent": "researcher", "expected_output": "Findings"},
         {"name": "analysis_task", "agent": "analyst", "expected_output": "Short report"}]
outputs = []
for task in tasks:
    assert task["agent"] in agents
    outputs.append({"task": task["name"], "input_from_previous": list(outputs), "result": task["expected_output"]})
print([item["task"] for item in outputs])
assert len(outputs[1]["input_from_previous"]) == 1

**Check:** No. A task assigns work to a role. This dictionary simulation does not execute CrewAI.

**Change one thing:** Add a reviewer role and task. Describe whether its input should include the analyst output.

## 2. Configuration fills named placeholders

The crew entry point supplies inputs, and YAML descriptions contain placeholders. Configuration names must match the inputs used at runtime.

**Predict:** What happens if motion is not supplied?

In [ ]:
description = "Propose an argument for: {motion}."
inputs = {"motion": "Beginners should practice offline first"}
rendered = description.format(**inputs)
print(rendered)
assert inputs["motion"] in rendered
try:
    description.format(**{})
except KeyError:
    print("Missing placeholder detected")

**Check:** Formatting raises KeyError here. Crew configuration and project environment are separate from the root project.

**Change one thing:** Add a second placeholder named audience and provide its value. Identify matching input keys in the linked source main.py.

## 3. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 4. Reducer vs. checkpoints vs. conversation identity

A reducer combines state updates. A checkpointer retains state across invocations. A thread_id selects a conversation. This uses SQLite in memory, so it creates no database file.

**Predict:** Does a different thread_id share the same list?

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
class MemoryState(TypedDict):
    turns: Annotated[list[str], add]
builder = StateGraph(MemoryState)
builder.add_node("reply", lambda state: {"turns": ["assistant reply"]})
builder.add_edge(START, "reply")
builder.add_edge("reply", END)
with SqliteSaver.from_conn_string(":memory:") as saver:
    graph = builder.compile(checkpointer=saver)
    config = {"configurable": {"thread_id": "practice-1"}}
    graph.invoke({"turns": ["first user turn"]}, config)
    result = graph.invoke({"turns": ["second user turn"]}, config)
    separate = graph.invoke({"turns": ["new user"]}, {"configurable": {"thread_id": "practice-2"}})
    print(result["turns"], separate["turns"])
    assert len(result["turns"]) == 4 and len(separate["turns"]) == 2

**Check:** No. Same ID continues a thread; a different ID selects another. In-memory SQLite here disappears when the connection closes.

**Change one thing:** Replace :memory: with a deliberately chosen practice database only if you want persistence. Explain when it would survive process exit.

## 5. Drafting and delivering are separate steps

An agent that formats text is different from a tool that sends email or notifications. This exercise records a delivery intention in memory only.

**Predict:** Will constructing this draft contact anyone?

In [ ]:
outbox = []
draft = {"to": "learner@example.com", "subject": "Practice report", "body": "Fictional findings."}
assert outbox == []
def dry_run_send(message):
    outbox.append({"status": "dry_run", "subject": message["subject"]})
    return {"status": "dry_run"}
result = dry_run_send(draft)
print(result, outbox)
assert len(outbox) == 1

**Check:** No. Neither this draft nor this in-memory dry run sends anything. A provider send/post call is the delivery boundary.

**Change one thing:** Format the body as HTML but keep dry_run_send unchanged. Explain where real delivery would begin.

## 6. Cache and simulation are not live prices

The trading exercises use market data and cached values. A fixed fictional quote lets you observe caching. A fallback simulation must stay identifiable as simulated data.

**Predict:** How many provider-fixture calls occur for the same cached symbol?

In [ ]:
from functools import lru_cache
calls = []
@lru_cache(maxsize=8)
def practice_quote(symbol):
    calls.append(symbol)
    return {"symbol": symbol, "price": 100.0, "source": "fictional fixture"}
first = practice_quote("DEMO")
second = practice_quote("DEMO")
print(first, "Fixture calls:", len(calls))
assert len(calls) == 1 and second["source"] == "fictional fixture"

**Check:** One. Cached or random fallback prices are not verified current market values.

**Change one thing:** Call a second symbol. Explain how you would record a quote timestamp and expire old cache entries.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.